# Financial Impact Forecasting Pipeline
This notebook demonstrates how to load text data over a mock timeline, process sequential inputs, and bench-mark **LSTM** vs **GRU** network performance.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

# Import custom modular components
from data_processing.text_cleaner import FinancialTextPreprocessor
from models.lstm_model import build_lstm_model
from models.gru_model import build_gru_model

print("Modules successfully imported!")

### 1. Generating Mock Financial Data
We create mock news items containing target sentiments over a dummy dates timeline.

In [ ]:
data = {
    'date': pd.date_range(start='2026-01-01', periods=100, freq='D'),
    'headline': [
        "Central Bank announces drastic interest rate hikes amid soaring inflation metrics.",
        "Tech startup values crash following catastrophic Q3 auditing reports.",
        "Forex market experiences steady consolidation over retail stability projections.",
        "Minor fluctuations reported within international regulatory bonds trading markets."
    ] * 25,
    'impact_level': [2, 2, 1, 0] * 25 # 0: LOW, 1: MEDIUM, 2: HIGH
}
df = pd.DataFrame(data)
df.head()

### 2. Preprocessing & Sequence Vectorization

In [ ]:
preprocessor = FinancialTextPreprocessor(max_words=1000, max_len=20)
X_padded = preprocessor.fit_transform(df['headline'])
y = df['impact_level'].values

X_train, X_test, y_train, y_test = train_test_split(X_padded, y, test_size=0.2, random_state=42)
vocab_size = len(preprocessor.tokenizer.word_index) + 1
print(f"Dataset ready. Vocabulary Size: {vocab_size}")

### 3. Model Compiling and Execution Comparison

In [ ]:
lstm = build_lstm_model(vocab_size=vocab_size, embedding_dim=32, input_length=20)
print("\n--- Training LSTM Model ---")
lstm.fit(X_train, y_train, epochs=3, batch_size=4, validation_split=0.1, verbose=1)

gru = build_gru_model(vocab_size=vocab_size, embedding_dim=32, input_length=20)
print("\n--- Training GRU Model ---")
gru.fit(X_train, y_train, epochs=3, batch_size=4, validation_split=0.1, verbose=1)